# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the mass spectrometry protein dataset described by a FAIR Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset metadata and structure are defined at the following Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if necessary
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect the main descriptive fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")
print(f"Version: {getattr(metadata, 'version', None)}\nLicense: {getattr(metadata, 'license', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}\nCitation: {getattr(metadata, 'citeAs', None)}")

## 2. Data Overview
Lists the dataset's record sets and their available fields. Each entity is referenced by its Croissant `@id`.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")

for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, data type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
We'll extract all records from each record set. Below we demonstrate loading the first record set by its `@id`.

For demonstration, we'll use the first discovered record set. You may change the `record_set_id` below to another found in the previous step.

In [ ]:
# Extract data from available record sets into DataFrames
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for RecordSet @id={rs.id} ({rs.name})\n")

# Pick the primary record set (most datasets of this type have only one main record set)
record_set_id = record_sets[0].id
df = dataframes[record_set_id]
print(f"Available columns in DataFrame loaded from @id={record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Now, we'll process numeric fields for outlier detection, normalization, and grouping. Choose a numeric field and a grouping field from the available DataFrame columns above. All columns are referenced by their Croissant `@id`.

In [ ]:
# Select numeric field and grouping field by examining columns
# Example: numeric fields may be 'cr:MW' (molecular weight), 'cr:pI', 'cr:Coverage', 'cr:Abundance', etc.
numeric_field = None
for col in df.columns:
    # Heuristically pick a likely numeric column such as 'cr:MW' (molecular weight)
    if 'MW' in col or 'Coverage' in col or 'Abundance' in col:
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0]
    print(f"Falling back to first numeric column: {numeric_field}")
print(f"Selected numeric field for filtering: {numeric_field}")

# Pick a group field
group_field = None
for col in df.columns:
    if 'Description' in col or 'id' in col or 'cr:description' in col:
        group_field = col
        break
if group_field is None and len(df.columns) > 1:
    group_field = df.columns[1]
print(f"Selected group field: {group_field}")

# Ensure the column is float for numeric operations (some CSV loads may load numbers as strings)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter records (e.g., MW > 10000)
threshold = 10000
filtered_df = df[df[numeric_field] > threshold]
print(f"Records where {numeric_field} > {threshold}: {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the numeric field
field_norm = f"{numeric_field}_normalized"
filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} as {field_norm}")
display(filtered_df[[numeric_field, field_norm]].head())

# Group by the group field (if meaningful)
if group_field in filtered_df.columns and pd.api.types.is_string_dtype(filtered_df[group_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and demonstrate possible relationships with other fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group (if enough groups exist and group_field is string/categorical)
if group_field in filtered_df.columns:
    n_unique = filtered_df[group_field].nunique()
    if 1 < n_unique <= 25:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load protein mass spectrometry data from a machine-readable Croissant schema.

- We inspected available record sets and fields (all referenced by their Croissant `@id`s).
- We extracted the main quantitative record set into a Pandas DataFrame.
- Performed filtering, normalization, simple grouping, and made quick visualizations on a core numeric field.

This approach enables reproducible, FAIR-compliant data loading and supports further statistical/proteomics analysis. For further work, explore additional quantitative columns or correlate modifications with abundance and coverage across experimental conditions.